In [1]:
# ==============================================================================
# ESTIMACIÓN DEL MODELO DE EFECTOS ALEATORIOS (RE) Y EXPORTACIÓN A LATEX
# ==============================================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
from linearmodels.panel import RandomEffects

# 1. Cargar y preparar datos
df = pd.read_csv('/home/hjvargaso/proyectos/tesis_sequia_2/data/processed/datos_mensuales_con_spei.csv')
df['fecha'] = pd.to_datetime(df['fecha'])

cluster_mapping = {
    'CAAZ': 1, 'PILAR': 1, 'MINGA': 1, 'COVIE': 1, 'CMEZA': 1,
    'LUQUE': 2, 'CONC': 2, 'SEST': 2, 'PJC': 2, 'MCAL': 2, 'GBRU': 2
}
df['cluster'] = df['estacion_id'].map(cluster_mapping)
df['tiempo'] = (df['fecha'].dt.year - 2000) * 12 + df['fecha'].dt.month

df['estacion_climatica'] = np.where(df['fecha'].dt.month.isin([12, 1, 2]), 'verano',
                           np.where(df['fecha'].dt.month.isin([3, 4, 5]), 'otono',
                           np.where(df['fecha'].dt.month.isin([9, 10, 11]), 'primavera', 'invierno')))
df['region_geografica'] = df['cluster'].astype(str)

df_panel = df.set_index(['estacion_id', 'fecha'])

# 2. Generar matrices de diseño
from patsy import dmatrices
formula = "spei_6 ~ 1 + tiempo * C(estacion_climatica, Treatment('invierno')) * C(region_geografica, Treatment('2'))"
y, X = dmatrices(formula, df_panel, return_type='dataframe')

# 3. Estimar el modelo final RE con errores estándar agrupados (clustered)
re_model = RandomEffects(y, X).fit(cov_type='clustered', cluster_entity=True)

# 4. Mostrar estadísticas generales en la consola
print(re_model)

# 5. GENERACIÓN DINÁMICA DE TABLAS EN LATEX (Ajustado para evitar advertencias y errores)
coef = re_model.params
std_err = re_model.std_errors
pvalues = re_model.pvalues
tstats = re_model.tstats

def get_stars(p):
    if p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.1: return '*'
    return ''

# A. Tabla de Parámetros en LaTeX
print("\n% ========================================================")
print("% CUADRO DE REGRESORES Y COEFICIENTES EN LATEX")
print("% ========================================================")
print("\\begin{table}[ht!]")
print("\\centering")
print("\\caption{Resultados del modelo de panel con efectos aleatorios para la variable de SPEI-6. Errores estándar agrupados por estación.}")
print("\\label{tab:resultados_re}")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("\\textbf{Regresor} & \\textbf{Coeficiente} & \\textbf{Error Est.} & \\textbf{Estadístico t} & \\textbf{p-valor} \\\\ \\hline")

for col in coef.index:
    name_clean = col.replace('_', '\\_').replace('%', '\\%')
    stars = get_stars(pvalues[col])
    print(f"{name_clean} & {coef[col]:.4f}{stars} & {std_err[col]:.4f} & {tstats[col]:.2f} & {pvalues[col]:.4f} \\\\")

print("\\hline")
print("\\multicolumn{5}{l}{\\small Nota: *** $p < 0.01$, ** $p < 0.05$, * $p < 0.1$. Categorías de referencia: Invierno y Clúster 2.} \\\\")
print("\\end{tabular}")
print("\\end{table}\n")

# B. Tabla de Estadísticas del Modelo en LaTeX (Corregido .pval y escapes)
print("% ========================================================")
print("% CUADRO DE ESTADÍSTICAS GENERALES EN LATEX")
print("% ========================================================")
print("\\begin{table}[ht!]")
print("\\centering")
print("\\caption{Estadísticas de ajuste del modelo de efectos aleatorios estimado.}")
print("\\label{tab:estadisticas_modelo}")
print(r"\begin{tabular}{lc}")
print("\\hline")
print(r"\textbf{Métrica de Ajuste} & \textbf{Valor del Estimador} \\ \hline")
print(f"Número de Observaciones ($N \\times T$) & {re_model.nobs} \\\\")
print(f"R-cuadrado global ($R^2$ overall) & {re_model.rsquared:.4f} \\\\")
print(f"R-cuadrado dentro ($R^2$ within) & {re_model.rsquared_within:.4f} \\\\")
print(f"R-cuadrado entre ($R^2$ between) & {re_model.rsquared_between:.4f} \\\\")
print(f"Estadístico Wald ($\\chi^2$) & {re_model.f_statistic.stat:.4f} \\\\")
print(f"p-valor de la regresión & {re_model.f_statistic.pval:.4f} \\\\")
print("\\hline")
print("\\end{tabular}")
print("\\end{table}\n")

                        RandomEffects Estimation Summary                        
Dep. Variable:                 spei_6   R-squared:                        0.0369
Estimator:              RandomEffects   R-squared (Between):              0.4116
No. Observations:                3113   R-squared (Within):               0.0369
Date:                Mon, Jun 01 2026   R-squared (Overall):              0.0369
Time:                        09:32:57   Log-likelihood                   -4358.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.9070
Entities:                          11   P-value                           0.0000
Avg Obs:                       283.00   Distribution:                 F(15,3097)
Min Obs:                       283.00                                           
Max Obs:                       283.00   F-statistic (robust):          3.198e+14
                            